In [ ]:
# 1
import requests
import pandas as pd

url = "https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json"
}

res = requests.get(url, headers=headers, timeout=10)
res.raise_for_status()

data = res.json()
tracks = data['response']['result']['chart']['items']['tracks']

results = []

for i, track in enumerate(tracks):
    rank = i + 1
    title = track.get('trackTitle')
    artists = [artist.get('artistName') for artist in track.get('artists', [])]
    
    results.append({
        '순위': rank,
        '곡명': title,
        '아티스트': artists
    })

print(f"수집된 데이터 건수: {len(results)}건")

df = pd.DataFrame(results)
df["아티스트"] = df["아티스트"].map(lambda x: ", ".join(x))
df.to_csv("vibe_top100.csv", index=False, encoding='utf-8-sig')

print(df.head())

수집된 데이터 건수: 100건
   순위           곡명           아티스트
0   1  LOVE ATTACK   RESCENE(리센느)
1   2          갑자기  아이오아이 (I.O.I)
2   3       REDRED  CORTIS (코르티스)
3   4      It's Me     아일릿(ILLIT)
4   5      여름아 부탁해         볼빨간사춘기


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time

url = "https://finance.naver.com/item/sise_day.naver"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

results = []
max_page = 40
prev_page_data = []
is_finished = False

one_year_ago = datetime.now() - timedelta(days=365)

for page in range(1, max_page + 1):
    params = {
        "code": "005930",
        "page": page
    }
    
    res = requests.get(url, headers=headers, params=params, timeout=10)
    res.raise_for_status()
    
    soup = BeautifulSoup(res.text, "html.parser")
    
    trs = soup.select("table.type2 tr")
    current_page_data = []
    
    for tr in trs:
        tds = tr.select("td")
        if len(tds) < 7 or not tds[0].text.strip():
            continue
            
        date_str = tds[0].text.strip()
        row_date = datetime.strptime(date_str, "%Y.%m.%d")
        
        if row_date < one_year_ago:
            is_finished = True
            break
            
        diff_td = tds[2]
        img = diff_td.find('img')
        diff_val = diff_td.text.strip().replace(',', '')
        
        if img and 'alt' in img.attrs:
            diff_text = f"{img['alt']} {diff_val}"
        else:
            diff_text = diff_val 
            
        try:
            close_val = int(tds[1].text.strip().replace(',', ''))
            open_val = int(tds[3].text.strip().replace(',', ''))
            high_val = int(tds[4].text.strip().replace(',', ''))
            low_val = int(tds[5].text.strip().replace(',', ''))
            vol_val = int(tds[6].text.strip().replace(',', ''))
        except ValueError:
            continue
            
        item = {
            '날짜': date_str,
            '종가': close_val,
            '전일비': diff_text,
            '시가': open_val,
            '고가': high_val,
            '저가': low_val,
            '거래량': vol_val
        }
        
        current_page_data.append(item)
        
    if not current_page_data:
        print(f"{page}페이지가 비어 있습니다. 종료.")
        break
        
    if current_page_data == prev_page_data:
        print(f"{page}페이지가 이전과 동일합니다. 종료.")
        break
        
    results.extend(current_page_data)
    prev_page_data = current_page_data
    
    print(f"{page}페이지 · 누적 {len(results)}건 · 최종 {current_page_data[-1]['날짜']}")
    
    if is_finished:
        print("1년 이전 날짜에 도달했습니다. 종료.")
        break
        
    time.sleep(0.5)

df = pd.DataFrame(results)
df.to_csv("samsung_1y.csv", index=False, encoding='utf-8-sig')
print(f"\n최종 {len(df)}건 저장 완료")

1페이지 · 누적 10건 · 최종 2026.08.07
2페이지 · 누적 20건 · 최종 2026.07.24
3페이지 · 누적 30건 · 최종 2026.07.09
4페이지 · 누적 40건 · 최종 2026.06.25
5페이지 · 누적 50건 · 최종 2026.06.11
6페이지 · 누적 60건 · 최종 2026.05.27
7페이지 · 누적 70건 · 최종 2026.05.12
8페이지 · 누적 80건 · 최종 2026.04.24
9페이지 · 누적 90건 · 최종 2026.04.10
10페이지 · 누적 100건 · 최종 2026.03.27
11페이지 · 누적 110건 · 최종 2026.03.13
12페이지 · 누적 120건 · 최종 2026.02.26
13페이지 · 누적 130건 · 최종 2026.02.09
14페이지 · 누적 140건 · 최종 2026.01.26
15페이지 · 누적 150건 · 최종 2026.01.12
16페이지 · 누적 160건 · 최종 2025.12.24
17페이지 · 누적 170건 · 최종 2025.12.10
18페이지 · 누적 180건 · 최종 2025.11.26
19페이지 · 누적 190건 · 최종 2025.11.12
20페이지 · 누적 200건 · 최종 2025.10.29
21페이지 · 누적 210건 · 최종 2025.10.15
22페이지 · 누적 220건 · 최종 2025.09.24
23페이지 · 누적 230건 · 최종 2025.09.10
24페이지 · 누적 240건 · 최종 2025.08.27
25페이지 · 누적 243건 · 최종 2025.08.22
1년 이전 날짜에 도달했습니다.

최종 243건 저장 완료


In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def crawl_naver_news(keyword, page):
    url = "https://search.naver.com/search.naver"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    
    results = []
    prev_page_data = []
    
    print(f"'{keyword}' 뉴스 수집을 시작합니다. (총 {page}페이지)")
    
    for p in range(1, page + 1):
        start = (p - 1) * 10 + 1
        
        params = {
            "ssc": "tab.news.all",
            "query": keyword,
            "start": start
        }
        
        res = requests.get(url, headers=headers, params=params, timeout=10)
        res.raise_for_status()
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        items = soup.select("div.fds-news-item-list-tab > div")
        
        if not items:
            print(f"{p}번째 스크롤(페이지)에서 결과가 없어 종료합니다.")
            break
            
        current_page_data = []
        
        for item in items:
            title_tag = item.select_one("a.news_tit")
            if not title_tag:
                title_span = item.select_one("span.sds-comps-text-type-headline1")
                if title_span:
                    title_tag = title_span.find_parent("a")
                    
            if not title_tag:
                continue
                
            title = title_tag.text.strip()
            link = title_tag.get("href")
            
            press_tag = item.select_one("a.info.press, span.sds-comps-profile-info-title-text")
            press = press_tag.text.replace("언론사 선정", "").strip() if press_tag else ""
            
            summary_tag = item.select_one(".news_dsc, .api_txt_lines.dsc_txt_wrap, span.sds-comps-text-type-body1")
            summary = summary_tag.text.strip() if summary_tag else ""
            
            current_page_data.append({
                "제목": title,
                "언론사": press,
                "링크": link,
                "요약": summary
            })
            
        if current_page_data == prev_page_data:
            print(f"{p}번째 스크롤(페이지)이 이전과 동일합니다.")
            break
            
        results.extend(current_page_data)
        prev_page_data = current_page_data
        
        print(f"{p}번째 스크롤(페이지) 수집 완료 · 누적 {len(results)}건")
        
        time.sleep(0.5)
        
    df = pd.DataFrame(results)
    filename = f"news_{keyword}.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    
    return results

crawled_data = crawl_naver_news("AI", 3)
print(f"총 {len(crawled_data)}건의 데이터가 'news_AI.csv' 파일로 저장되었습니다.")

'AI' 뉴스 수집을 시작합니다. (총 3페이지)
1번째 스크롤(페이지) 수집 완료 · 누적 10건
2번째 스크롤(페이지) 수집 완료 · 누적 20건
3번째 스크롤(페이지) 수집 완료 · 누적 30건
총 30건의 데이터가 'news_AI.csv' 파일로 저장되었습니다.
